In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# What is this notebook about?

When fitting the VAR, we get some matmul errors, so I took it upon myself to check whether there was actually any issues with the matrix mutiplication and did it step by step. Turns out that for whatever reason python is giving some warning when trying to multiply $Z'Y$ Howver, this warning isn't affecting the estimated coefficients at all, as it is possible to check in this code, comparing the `betas` matrix with the coefficients estimates by the statsmodels VAR

In [2]:
# Load all the numeric data
source = Path('../source/')
indpro = pd.read_excel(source / 'INDPRO.xlsx', sheet_name= 'Monthly')
unrate = pd.read_excel(source / 'UNRATE.xlsx', sheet_name= 'Monthly')
dff = pd.read_excel(source / 'FEDFUNDS.xlsx', sheet_name= 'Monthly')
cpi = pd.read_excel(source / 'CPIAUCSL.xlsx', sheet_name= 'Monthly')
epu = pd.read_excel(source / 'USEPUINDXD.xlsx', sheet_name= 'Monthly')
vix = pd.read_excel(source / 'VIXCLS.xlsx', sheet_name= 'Monthly')

# Join everything into a single df
data = indpro.merge(unrate, on = 'observation_date', how= 'outer')
data = data.merge(dff, on = 'observation_date', how = 'outer')
data = data.merge(cpi, on = 'observation_date', how = 'outer')
data = data.merge(epu, on = 'observation_date', how = 'outer')
data = data.merge(vix, on = 'observation_date', how = 'outer')

data.sort_values(by = 'observation_date', inplace= True)
data.columns = [col.lower() for col in data.columns] # Rename all columns to lowercase

del indpro, unrate, dff, cpi, epu, vix, source

In [ ]:
data['usepuindxd'] = (data['usepuindxd']/100) + 1 
apply_logs = ['cpiaucsl', 'indpro', 'vixcls', 'usepuindxd'] # Columns we'll apply logs to
log_cols = [f'log_{col}' for col in apply_logs] # Names of the log columns
data[log_cols] = np.log(data[apply_logs]) # Apply logs to the columns

# Compute log differences and plain differences
data["inflation"]    = 1200.0 * (data["log_cpiaucsl"] - data["log_cpiaucsl"].shift(1))
data["delta_indpro"] = 1200.0 * (data["log_indpro"]   - data["log_indpro"].shift(1))

del apply_logs, log_cols

In [6]:
def check_numerical_stability(data: pd.DataFrame, lag_order: int, scale: bool = True):
    '''This function takes some input data and a lag order, 
    constructs the Z and Y matrices for a VAR model and checks the numerical stability of the Z and Z'Z matrices.
    Finally, it estimates the VAR coefficients using OLS and returns Z, Y and the betas 
    '''
    if scale:
        scaler = StandardScaler()
        data = pd.DataFrame(scaler.fit_transform(data), columns= data.columns, index= data.index)

    Y = data.iloc[lag_order:].to_numpy(dtype=np.float64)
    Z_parts = [np.ones((len(data) - lag_order, 1), dtype = np.float64)]
    regressor_names = ['const']
    for lag in range(1, lag_order + 1):
        Z_parts.append(data.shift(lag).iloc[lag_order:].to_numpy(dtype=np.float64))
        for col in data.columns:
            regressor_names.append(f'{col}_lag{lag}')
    Z = np.hstack(Z_parts)
    print("Z shape:", Z.shape)
    print("rank(Z):", np.linalg.matrix_rank(Z), "out of", Z.shape[1])
    print("cond(Z):", np.linalg.cond(Z))

    ZTY = np.einsum('ti,tj->ij', Z, Y)
    ZTZ = Z.T @ Z
    print("cond(Z'Z):", np.linalg.cond(ZTZ))

    eigvals = np.linalg.eigvalsh(ZTZ)
    print("Min eigenvalue Z'Z:", np.min(eigvals))
    print("Max eigenvalue Z'Z:", np.max(eigvals))

    betas = np.linalg.solve(ZTZ, ZTY)
    betas = pd.DataFrame(betas, columns= data.columns, index = regressor_names)
    return Z, Y, betas

In [7]:
stationary_columns = ['inflation', 'delta_indpro', 'unrate', 'log_vixcls', 'log_usepuindxd', 'fedfunds']
stationary_data = data[stationary_columns].dropna().reset_index(drop = True).copy()

In [ ]:
Z, Y, betas = check_numerical_stability(stationary_data, lag_order= 4)

Z shape: (428, 25)
rank(Z): 25 out of 25
cond(Z): 117.30584215928442
cond(Z'Z): 13760.660604666138
Min eigenvalue Z'Z: 0.21735931506464826
Max eigenvalue Z'Z: 2991.0077638679923


In [9]:
model = VAR(stationary_data)
lag_order = model.select_order(maxlags=12)
print(lag_order.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0        4.591       4.649       98.63       4.614
1       -6.586      -6.182    0.001380      -6.426
2       -7.261     -6.511*   0.0007022     -6.965*
3       -7.301      -6.204   0.0006753      -6.867
4      -7.335*      -5.892  0.0006528*      -6.765
5       -7.282      -5.492   0.0006891      -6.575
6       -7.248      -5.113   0.0007133      -6.404
7       -7.167      -4.685   0.0007753      -6.186
8       -7.097      -4.269   0.0008329      -5.979
9       -7.046      -3.872   0.0008785      -5.792
10      -7.120      -3.600   0.0008185      -5.729
11      -7.065      -3.197   0.0008691      -5.536
12      -7.052      -2.838   0.0008846      -5.386
--------------------------------------------------


In [10]:
results = model.fit(4)
results.summary()

/Users/md49543/Library/Python/3.13/lib/python/site-packages/statsmodels/tsa/vector_ar/var_model.py:1501: RuntimeWarning: divide by zero encountered in matmul
  return np.kron(np.linalg.inv(z.T @ z), self.sigma_u)
/Users/md49543/Library/Python/3.13/lib/python/site-packages/statsmodels/tsa/vector_ar/var_model.py:1501: RuntimeWarning: overflow encountered in matmul
  return np.kron(np.linalg.inv(z.T @ z), self.sigma_u)
/Users/md49543/Library/Python/3.13/lib/python/site-packages/statsmodels/tsa/vector_ar/var_model.py:1501: RuntimeWarning: invalid value encountered in matmul
  return np.kron(np.linalg.inv(z.T @ z), self.sigma_u)


  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 14, Apr, 2026
Time:                     21:01:38
--------------------------------------------------------------------
No. of Equations:         6.00000    BIC:                   -5.91140
Nobs:                     428.000    HQIC:                  -6.77214
Log likelihood:          -1924.36    FPE:                0.000653488
AIC:                     -7.33399    Det(Omega_mle):     0.000464848
--------------------------------------------------------------------
Results for equation inflation
                       coefficient       std. error           t-stat            prob
------------------------------------------------------------------------------------
const                     1.213493         1.445227            0.840           0.401
L1.inflation              0.499523         0.049958            9.999           0.000
L1.delta_indpro           0.016283      